# Riccati: a nonlinear first-order ODE

**Book:** §3.1 &nbsp;·&nbsp; `ch03/riccati_ode.ipynb`

$$\frac{du}{dt} = q_0 + q_1 u + q_2 u^2 \quad\xrightarrow{\;q_0=1,\;q_1=0,\;q_2=-1\;}\quad
u' = 1-u^2,\qquad u(0)=0,$$

with exact solution $u(t)=\tanh t$ on $t\in[0,4]$.

Trial function $u=t\,\mathcal N(t)$ makes $u(0)=0$ exact, so again a single-term loss

$$\mathcal{L}=\frac{1}{N_f}\sum_i \big(u'(t_i) - 1 + u(t_i)^2\big)^2 .$$

**The point:** the quadratic term $u^2$ costs *nothing*. No linearisation, no Jacobian, no Newton
loop — autograd differentiates the Python expression as written.

In [ ]:
import time
import numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0)

def g1(f, x): return torch.autograd.grad(f, x, torch.ones_like(f), create_graph=True)[0]

T = 4.0
net  = nn.Sequential(nn.Linear(1,32), nn.Tanh(), nn.Linear(32,32), nn.Tanh(), nn.Linear(32,1))
u_of = lambda t: t*net(t/T)                      # u(0)=0 exactly; input normalised
opt  = torch.optim.Adam(net.parameters(), 3e-3)

t0 = time.perf_counter()
for e in range(5000):
    if e == 3500:
        for g in opt.param_groups: g['lr'] = 5e-4
    opt.zero_grad()
    t = (torch.rand(256,1)*T).requires_grad_(True)
    u = u_of(t)
    res = g1(u, t) - 1 + u**2                    # the nonlinearity, written as written
    (res**2).mean().backward(); opt.step()
print(f'training: {time.perf_counter()-t0:.1f} s')

tg = torch.linspace(0, T, 400).reshape(-1,1)
with torch.no_grad(): up = u_of(tg).numpy().ravel()
te = np.tanh(tg.numpy().ravel())
err = np.sqrt(np.mean((up-te)**2)/np.mean(te**2))
print(f'relative L2 error: {err:.1e}')

plt.figure(figsize=(7,4))
plt.plot(tg.numpy(), te, 'g', lw=2.6, alpha=.6, label=r'exact $\tanh t$')
plt.plot(tg.numpy(), up, 'r--', lw=1.6, label=f'PINN (rel $L_2$={err:.1e})')
plt.xlabel('t'); plt.ylabel('u'); plt.legend(); plt.grid(alpha=.3)
plt.title("Riccati $u' = 1 - u^2$: the nonlinearity is free")
plt.tight_layout(); plt.show()